# 🔬 Blood Cell Segmentation — Preprocessing Pipeline
**Pipeline**: Denoise → Color Space → Global Channel Config → WBC Seg → RBC Seg → Overlay

> **Cách dùng:**
> 1. Chạy **Cell 1–3** (setup)
> 2. Chạy **Cell 4** một lần để học channel config từ train+val set
> 3. Chạy **Cell 5** để load config
> 4. Chạy **Cell 6** để xử lý ảnh

In [43]:
import os
import cv2
import json
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from glob import glob
from scipy import ndimage
from skimage import morphology
import warnings

matplotlib.use('Agg')
warnings.filterwarnings('ignore')

In [44]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_PATH    = '/kaggle/input/datasets/orvile/bccd-blood-cell-count-and-detection-dataset'
CHANNEL_CONFIG_PATH = '/kaggle/working/channel_config.json'
PIPELINE_FIG_PATH   = '/kaggle/working/blood_cell_pipeline.png'

SAMPLE_IMG = os.path.join(DATASET_PATH, 'train', 'img', 'BloodImage_00001.jpeg')
SAMPLE_ANN = os.path.join(DATASET_PATH, 'train', 'ann', 'BloodImage_00001.jpeg.json')

# Preview ảnh mẫu
sample_bgr = cv2.imread(SAMPLE_IMG)
sample_rgb = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(6, 4))
plt.title('Sample Image')
plt.imshow(sample_rgb)
plt.axis('off')
plt.show()

## Section A — Core Pipeline Functions
Utility, Denoise, Color Space, Contrast, WBC/RBC Segmentation, Visualization

In [45]:
# ============================================================
# SECTION 0: UTILITY & HELPER FUNCTIONS
# ============================================================

def adaptive_kernel_size(image: np.ndarray) -> int:
    """Tự động tính kernel size (luôn là số lẻ) dựa trên kích thước ảnh."""
    h, w = image.shape[:2]
    k = max(3, int(max(h, w) / 150))
    return k + 1 if k % 2 == 0 else k


def compute_gamma(image: np.ndarray) -> float:
    """Tự động tính gamma: tối → làm sáng (< 1), sáng → làm tối (> 1)."""
    mean = np.mean(image)
    if mean < 100:   return 0.6
    elif mean > 155: return 1.5
    else:            return 1.0


def apply_gamma_correction(image: np.ndarray, gamma: float) -> np.ndarray:
    """Gamma correction qua lookup table (nhanh hơn pow trực tiếp)."""
    table = np.array([((i / 255.0) ** (1.0 / gamma)) * 255
                      for i in np.arange(256)], dtype=np.uint8)
    return cv2.LUT(image, table)


def apply_sharpening(image: np.ndarray, strength: float = 1.5) -> np.ndarray:
    """Unsharp Masking: Original + strength * (Original - GaussianBlur)."""
    blurred = cv2.GaussianBlur(image, (0, 0), 3)
    return cv2.addWeighted(image, 1 + strength, blurred, -strength, 0)


def fill_holes(binary_mask: np.ndarray) -> np.ndarray:
    """Lấp đầy lỗ hổng bên trong các vùng trắng của binary mask."""
    return ndimage.binary_fill_holes(binary_mask.astype(bool)).astype(np.uint8) * 255


def remove_small_objects(binary_mask: np.ndarray, min_size: int = None) -> np.ndarray:
    """Xóa các vật thể nhỏ hơn min_size pixels."""
    if min_size is None:
        min_size = int(np.pi * (20 ** 2) * 0.1)
    return morphology.remove_small_objects(
        binary_mask.astype(bool), min_size=min_size
    ).astype(np.uint8) * 255


def create_synthetic_blood_image(size: int = 512) -> np.ndarray:
    """
    Tạo ảnh máu tổng hợp để demo khi không có ảnh thật.
    - WBC: tím/xanh đậm, lớn
    - RBC: đỏ/hồng, nhỏ, biconcave
    - Nền: be nhạt + noise
    """
    img = np.ones((size, size, 3), dtype=np.uint8)
    img[:, :] = [220, 210, 200]
    noise = np.random.normal(0, 8, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    rng = np.random.default_rng(42)
    rbc_centers = []

    for _ in range(55):
        for attempt in range(50):
            cx, cy, r = rng.integers(30, size-30), rng.integers(30, size-30), rng.integers(14, 22)
            if all(np.sqrt((cx-ox)**2 + (cy-oy)**2) >= r+or_+3
                   for ox, oy, or_ in rbc_centers):
                rbc_centers.append((cx, cy, r))
                break

    for cx, cy, r in rbc_centers:
        for dr in range(r, -1, -1):
            intensity = int(180 + 70 * (1 - dr / r))
            cr = min(255, intensity + int(rng.integers(-10, 10)))
            cg = max(0, int(intensity * 0.45) + int(rng.integers(-10, 10)))
            cb = max(0, int(intensity * 0.45) + int(rng.integers(-10, 10)))
            cv2.circle(img, (cx, cy), dr, (cb, cg, cr), -1)

    wbc_centers = []
    for _ in range(5):
        for attempt in range(100):
            cx, cy, r = rng.integers(60, size-60), rng.integers(60, size-60), rng.integers(28, 40)
            if all(np.sqrt((cx-ox)**2 + (cy-oy)**2) >= r+or_+5
                   for ox, oy, or_ in rbc_centers + wbc_centers):
                wbc_centers.append((cx, cy, r))
                break

    for cx, cy, r in wbc_centers:
        cv2.circle(img, (cx, cy), r,              (200, 170, 210), -1)
        cv2.circle(img, (cx, cy), int(r*0.65),   (120,  60, 160), -1)
        cv2.circle(img, (cx, cy), int(r*0.33),   ( 80,  30, 120), -1)

    return cv2.GaussianBlur(img, (3, 3), 0.5)


# ============================================================
# SECTION 1: DENOISING
# ============================================================

def denoise_image(image: np.ndarray, kernel_size: int = None) -> dict:
    """
    So sánh 4 phương pháp khử nhiễu và chọn tốt nhất tự động:
    Gaussian | Median | Bilateral | Non-local Means
    Tiêu chí: edge preservation * 0.6 + smoothness * 0.4
    """
    if kernel_size is None:
        kernel_size = adaptive_kernel_size(image)
    if kernel_size % 2 == 0:
        kernel_size += 1

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()

    candidates = {
        "gaussian":  cv2.GaussianBlur(image, (kernel_size, kernel_size), 0),
        "median":    cv2.medianBlur(image, kernel_size),
        "bilateral": cv2.bilateralFilter(image, 9, 75, 75),
        "nlm":       cv2.fastNlMeansDenoisingColored(image, None, 10, 10, 7, 21),
    }

    orig_edges = cv2.Canny(gray, 30, 100)
    best_method, best_score = "bilateral", -1.0

    for name, denoised in candidates.items():
        d_gray  = cv2.cvtColor(denoised, cv2.COLOR_BGR2GRAY)
        d_edges = cv2.Canny(d_gray, 30, 100)
        edge_overlap = float(np.logical_and(orig_edges, d_edges).sum()) / (orig_edges.sum() + 1e-8)
        smoothness   = 1.0 - float(cv2.Laplacian(d_gray, cv2.CV_64F).var()) / (laplacian_var + 1e-8)
        score = edge_overlap * 0.6 + max(0.0, min(1.0, smoothness)) * 0.4
        if score > best_score:
            best_score, best_method = score, name

    print(f"  [Denoise] Noise level: {laplacian_var:.1f} | Best: {best_method} (score={best_score:.3f})")
    return {**candidates,
            "best": candidates[best_method],
            "best_method_name": best_method,
            "noise_level": laplacian_var}


# ============================================================
# SECTION 2: COLOR SPACE ANALYSIS
# ============================================================

def analyze_color_spaces(image: np.ndarray) -> dict:
    """
    Trích xuất 12 channels từ 4 color spaces:
    RGB(R,G,B) | HSV(H,S,V) | LAB(L,Lab_A,Lab_B) | YCrCb(Y,Cr,Cb)
    """
    rgb   = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv   = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lab   = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)

    channels = {
        "R": rgb[:,:,0],   "G": rgb[:,:,1],   "B": rgb[:,:,2],
        "H": hsv[:,:,0],   "S": hsv[:,:,1],   "V": hsv[:,:,2],
        "L": lab[:,:,0],   "Lab_A": lab[:,:,1], "Lab_B": lab[:,:,2],
        "Y": ycrcb[:,:,0], "Cr": ycrcb[:,:,1], "Cb": ycrcb[:,:,2],
    }
    return {"channels": channels, "rgb": rgb, "hsv": hsv, "lab": lab, "ycrcb": ycrcb}


# ============================================================
# SECTION 3: CONTRAST ENHANCEMENT
# ============================================================

def enhance_contrast(channel: np.ndarray) -> dict:
    """
    Tăng tương phản bằng CLAHE + Unsharp Masking (tốt nhất cho ảnh kính hiển vi).
    Trả về: histogram_eq | clahe | gamma | clahe_sharp (dùng chính)
    """
    ch = channel.astype(np.uint8)
    clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_res  = clahe_obj.apply(ch)
    gamma      = compute_gamma(ch)

    print(f"  [Contrast] Gamma tự động: {gamma:.2f} | CLAHE clipLimit=2.0 tileGrid=(8,8)")
    return {
        "histogram_eq": cv2.equalizeHist(ch),
        "clahe":        clahe_res,
        "gamma":        apply_gamma_correction(ch, gamma),
        "clahe_sharp":  apply_sharpening(clahe_res, strength=1.2),
        "gamma_value":  gamma,
    }


# ============================================================
# SECTION 4: WBC SEGMENTATION
# ============================================================

def segment_wbc(channel: np.ndarray, kernel_size: int = None) -> dict:
    """
    Pipeline WBC: CLAHE → Smooth → Invert? → Otsu → Morphology → Fill → Size filter
    WBC có nhân đậm, kích thước lớn hơn RBC (~15-20µm vs 6-8µm).
    """
    if kernel_size is None:
        kernel_size = adaptive_kernel_size(channel)
    if kernel_size % 2 == 0:
        kernel_size += 1

    ch = channel.astype(np.uint8)

    # CLAHE + smooth
    clahe_ch  = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(ch)
    smoothed  = cv2.GaussianBlur(clahe_ch, (5, 5), 0)

    # WBC thường tối hơn nền → invert nếu dark_ratio < 0.5
    mean_total = np.mean(smoothed)
    ch_thresh  = (cv2.bitwise_not(smoothed)
                  if float(np.sum(smoothed < mean_total)) / smoothed.size < 0.5
                  else smoothed)

    # Otsu threshold
    otsu_thresh, otsu_mask = cv2.threshold(
        ch_thresh, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptive_mask = cv2.adaptiveThreshold(
        ch_thresh, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, blockSize=31, C=-2)

    print(f"  [WBC Seg] Otsu threshold: {otsu_thresh:.1f}")

    # Morphology
    k_open  = max(3, kernel_size - 2) | 1
    k_close = (kernel_size + 2) | 1
    kernel_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open,  k_open))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close))
    opened = cv2.morphologyEx(otsu_mask, cv2.MORPH_OPEN,  kernel_open,  iterations=2)
    closed = cv2.morphologyEx(opened,    cv2.MORPH_CLOSE, kernel_close, iterations=2)
    filled = fill_holes(closed)

    # Size filter: loại noise nhỏ và artifact quá lớn
    h, w = ch.shape
    min_wbc = int(np.pi * 20 * 20 * 0.3)
    max_wbc = int(h * w * 0.15)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(filled, connectivity=8)
    final_mask = np.zeros_like(filled)
    for lid in range(1, n_labels):
        area = stats[lid, cv2.CC_STAT_AREA]
        if min_wbc <= area <= max_wbc:
            final_mask[labels == lid] = 255

    wbc_count = cv2.connectedComponentsWithStats(final_mask, connectivity=8)[0] - 1
    print(f"  [WBC Seg] Phát hiện: {wbc_count} WBC")

    return {
        "clahe": clahe_ch, "smoothed": smoothed,
        "prepared_for_threshold": ch_thresh,
        "otsu_threshold": otsu_thresh, "otsu_mask": otsu_mask,
        "adaptive_mask": adaptive_mask,
        "morpho_open": opened, "morpho_close": closed,
        "filled": filled, "final_mask": final_mask,
    }


# ============================================================
# SECTION 5: RBC SEGMENTATION
# ============================================================

def _score_rbc_mask(mask: np.ndarray) -> dict:
    """
    Đánh giá chất lượng mask RBC qua 3 tiêu chí:
    - count      : số RBC phát hiện (nhiều hơn = tốt hơn)
    - circularity: RBC là đĩa tròn → circularity = 4π·area/perimeter² ≈ 1
    - uniformity : các RBC có kích thước đều nhau → std/mean thấp = tốt hơn

    Score tổng hợp (0–1): count * 0.4 + circularity * 0.4 + uniformity * 0.2
    """
    n_labels, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    count = n_labels - 1
    if count == 0:
        return {"count": 0, "circularity": 0.0, "uniformity": 0.0, "total": 0.0}

    areas = [stats[i, cv2.CC_STAT_AREA] for i in range(1, n_labels)]

    # Circularity trung bình: dùng contour của từng blob
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    circ_vals = []
    for cnt in contours:
        area  = cv2.contourArea(cnt)
        peri  = cv2.arcLength(cnt, True)
        if peri > 0:
            circ_vals.append(4 * np.pi * area / (peri ** 2))
    circularity = float(np.mean(circ_vals)) if circ_vals else 0.0

    # Uniformity: 1 - (std / mean), clip 0-1
    mean_area = float(np.mean(areas))
    std_area  = float(np.std(areas))
    uniformity = max(0.0, 1.0 - std_area / (mean_area + 1e-8))

    # Normalize count: cap ở 200 RBC (bình thường trong ảnh BCCD)
    count_norm = min(count / 200.0, 1.0)

    total = count_norm * 0.4 + circularity * 0.4 + uniformity * 0.2
    return {"count": count, "circularity": circularity,
            "uniformity": uniformity, "total": total}


def segment_rbc(image_no_wbc: np.ndarray, channel: np.ndarray,
                kernel_size: int = None) -> dict:
    """
    Pipeline RBC (sau khi đã loại WBC):
    CLAHE → Invert → Otsu → Morphology → Size filter → Watershed
    """
    if kernel_size is None:
        kernel_size = adaptive_kernel_size(channel)
    if kernel_size % 2 == 0:
        kernel_size += 1

    ch = channel.astype(np.uint8)

    # CLAHE + smooth + invert
    clahe_ch = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8)).apply(ch)
    smoothed = cv2.GaussianBlur(clahe_ch, (5, 5), 0)
    inverted = cv2.bitwise_not(smoothed)

    # Otsu
    otsu_thresh, otsu_mask = cv2.threshold(
        inverted, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    print(f"  [RBC Seg] Otsu threshold: {otsu_thresh:.1f}")

    # Morphology
    k_small = max(3, kernel_size - 4) | 1
    kernel_s = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_small, k_small))
    kernel_m = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    opened = cv2.morphologyEx(otsu_mask, cv2.MORPH_OPEN,  kernel_s, iterations=1)
    closed = cv2.morphologyEx(opened,    cv2.MORPH_CLOSE, kernel_m, iterations=1)
    filled = fill_holes(closed)

    # Size filter
    h, w = ch.shape
    min_rbc = int(np.pi * 8  * 8  * 0.4)
    max_rbc = int(np.pi * 35 * 35)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(filled, connectivity=8)
    rbc_mask = np.zeros_like(filled)
    rbc_count = 0
    for lid in range(1, n_labels):
        area = stats[lid, cv2.CC_STAT_AREA]
        if min_rbc <= area <= max_rbc:
            rbc_mask[labels == lid] = 255
            rbc_count += 1
    print(f"  [RBC Seg] Sau size filter: {rbc_count} RBC")

    # Watershed
    dist = cv2.distanceTransform(rbc_mask, cv2.DIST_L2, 5)
    cv2.normalize(dist, dist, 0, 1.0, cv2.NORM_MINMAX)
    _, sure_fg = cv2.threshold(dist, 0.4, 1.0, cv2.THRESH_BINARY)
    sure_fg  = np.uint8(sure_fg * 255)
    sure_bg  = cv2.dilate(rbc_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7)), iterations=3)
    unknown  = cv2.subtract(sure_bg, sure_fg)
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    img_ws = image_no_wbc.copy()
    if len(img_ws.shape) == 2:
        img_ws = cv2.cvtColor(img_ws, cv2.COLOR_GRAY2BGR)
    markers_ws = cv2.watershed(img_ws, markers.copy())

    watershed_mask = np.zeros(ch.shape, dtype=np.uint8)
    watershed_mask[markers_ws > 1]  = 255
    watershed_mask[markers_ws == -1] = 0
    final_rbc = cv2.bitwise_and(watershed_mask, rbc_mask)

    return {
        "clahe": clahe_ch, "otsu_mask": otsu_mask,
        "morpho": closed, "filtered_mask": rbc_mask,
        "distance_transform": dist,
        "watershed_markers": markers_ws, "watershed_mask": watershed_mask,
        "final_mask": final_rbc,
    }


# ============================================================
# SECTION 6: WBC OVERLAY (loại WBC khỏi ảnh trước khi segment RBC)
# ============================================================

def create_wbc_overlay(image: np.ndarray, wbc_mask: np.ndarray,
                       dilation_px: int = 3) -> dict:
    """
    Dilate mask WBC rồi inpaint để loại WBC khỏi ảnh.
    Trả về ảnh đã inpaint dùng cho RBC segmentation.
    """
    kernel_dil = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (dilation_px*2+1, dilation_px*2+1))
    dilated = cv2.dilate(wbc_mask, kernel_dil, iterations=2)

    overlay_black   = image.copy()
    overlay_black[dilated > 0] = [0, 0, 0]

    overlay_colored = image.copy()
    overlay_colored[wbc_mask > 0] = (overlay_colored[wbc_mask > 0] * 0.3).astype(np.uint8)
    overlay_colored[wbc_mask > 0, 2] = 200

    inpainted = cv2.inpaint(image, dilated, 5, cv2.INPAINT_TELEA)

    return {
        "wbc_mask_dilated": dilated,
        "overlay_black":    overlay_black,
        "overlay_colored":  overlay_colored,
        "inpainted":        inpainted,
        "image_no_wbc":     inpainted,
    }


# ============================================================
# SECTION 7: FINAL OVERLAY
# ============================================================

def create_final_overlay(original: np.ndarray,
                         wbc_mask: np.ndarray,
                         rbc_mask: np.ndarray) -> np.ndarray:
    """WBC → đỏ bán trong suốt | RBC → xanh lá bán trong suốt + contour."""
    overlay = original.copy()

    wbc_layer = original.copy()
    wbc_layer[wbc_mask > 0] = [0, 0, 220]
    overlay = cv2.addWeighted(overlay, 0.6, wbc_layer, 0.4, 0)

    rbc_layer = overlay.copy()
    rbc_layer[rbc_mask > 0] = [0, 200, 0]
    overlay = cv2.addWeighted(overlay, 0.7, rbc_layer, 0.3, 0)

    contours_wbc, _ = cv2.findContours(wbc_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours_rbc, _ = cv2.findContours(rbc_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours_wbc, -1, (0,   0, 255), 2)
    cv2.drawContours(overlay, contours_rbc, -1, (0, 255,   0), 1)

    return overlay


# ============================================================
# SECTION 8: VISUALIZATION
# ============================================================

def visualize_pipeline(results: dict, save_path: str = PIPELINE_FIG_PATH):
    """Grid 6×5 hiển thị toàn bộ pipeline: Preproc | Channels | WBC | RBC | Final | Stats."""
    fig = plt.figure(figsize=(24, 30))
    fig.patch.set_facecolor('#1a1a2e')
    TC, LC = '#e0e0ff', '#aaaadd'

    def add_img(ax, img, title, cmap='gray'):
        if img is None:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', color=LC, fontsize=10)
        elif len(img.shape) == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap=cmap)
        ax.set_title(title, color=TC, fontsize=9, fontweight='bold', pad=5)
        ax.axis('off')
        ax.set_facecolor('#0d0d1a')

    gs = gridspec.GridSpec(6, 5, figure=fig, hspace=0.35, wspace=0.08,
                           left=0.04, right=0.97, top=0.96, bottom=0.02)

    def row_axes(row): return [fig.add_subplot(gs[row, i]) for i in range(5)]
    def row_label(y, txt, color='#aaaadd', bg='#2a2a3e'):
        fig.text(0.01, y, txt, color=color, fontsize=7, va='center', ha='left',
                 bbox=dict(boxstyle='round', facecolor=bg, alpha=0.8))

    orig     = results.get("original")
    denoised = results.get("denoised")
    channels = results.get("channels", {})
    wbc_seg  = results.get("wbc_seg", {})
    rbc_seg  = results.get("rbc_seg", {})
    ovl      = results.get("overlay_data", {})

    # Row 1: Preprocessing steps
    ax = row_axes(0)
    add_img(ax[0], orig, "Ảnh gốc")
    add_img(ax[1], denoised, f"Denoise({results.get('denoise_method','bilateral')})")
    add_img(ax[2], results.get("clahe_wbc"), "CLAHE WBC", 'gray')
    add_img(ax[3], results.get("best_wbc_ch_img"), f"WBC Channel({results.get('best_wbc_name','B')})", 'gray')
    add_img(ax[4], results.get("best_rbc_ch_img"), f"RBC Channel({results.get('best_rbc_name','Cr')})", 'gray')
    row_label(0.925, "ROW 1\nPREPROC")

    # Row 2: Key channels
    ax = row_axes(1)
    for i, ch in enumerate(["R", "G", "B", "Lab_A", "Cr"]):
        add_img(ax[i], channels.get(ch), f"Channel {ch}", 'gray')
    row_label(0.770, "ROW 2\nCHANNELS")

    # Row 3: WBC segmentation steps
    ax = row_axes(2)
    add_img(ax[0], wbc_seg.get("clahe"),        "WBC CLAHE",          'gray')
    add_img(ax[1], wbc_seg.get("otsu_mask"),    "Otsu (WBC)",         'gray')
    add_img(ax[2], wbc_seg.get("morpho_close"), "Morphology",         'gray')
    add_img(ax[3], wbc_seg.get("filled"),       "Fill Holes",         'gray')
    add_img(ax[4], wbc_seg.get("final_mask"),   "WBC Final Mask",     'gray')
    row_label(0.615, "ROW 3\nWBC SEG", '#ff6b6b', '#3e2a2a')

    # Row 4: WBC overlay + RBC steps
    ax = row_axes(3)
    add_img(ax[0], ovl.get("overlay_colored"),  "WBC Highlight")
    add_img(ax[1], ovl.get("inpainted"),        "WBC Removed")
    add_img(ax[2], rbc_seg.get("otsu_mask"),    "Otsu (RBC)",         'gray')
    add_img(ax[3], rbc_seg.get("distance_transform"), "Distance Transform", 'hot')
    add_img(ax[4], rbc_seg.get("watershed_mask"),     "Watershed",          'gray')
    row_label(0.460, "ROW 4\nRBC SEG", '#6bff6b', '#2a3e2a')

    # Row 5: Final results
    ax = row_axes(4)
    add_img(ax[0], rbc_seg.get("morpho"),        "RBC Morphology",    'gray')
    add_img(ax[1], rbc_seg.get("filtered_mask"), "RBC Size Filter",   'gray')
    add_img(ax[2], results.get("wbc_mask"),      "WBC Final",         'RdPu')
    add_img(ax[3], results.get("rbc_mask"),      "RBC Final",         'Reds')
    add_img(ax[4], results.get("overlay"),       "Final Overlay")
    row_label(0.305, "ROW 5\nFINAL", '#ffd700', '#3e3a2a')

    # Row 6: Histograms
    ax = row_axes(5)
    hist_cfg = [
        (orig,                           "Ảnh gốc",   '#4a9eff'),
        (denoised,                       "Denoise",    '#ff9f4a'),
        (results.get("best_wbc_ch_img"), f"WBC ch ({results.get('best_wbc_name','')})", '#a06bff'),
        (results.get("best_rbc_ch_img"), f"RBC ch ({results.get('best_rbc_name','')})", '#ff6b6b'),
    ]
    for i, (img, label, color) in enumerate(hist_cfg):
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img
            ax[i].hist(gray.ravel(), 256, [0,256], color=color, alpha=0.7)
            ax[i].set_title(f"Histogram: {label}", color=TC, fontsize=9, fontweight='bold')
            ax[i].set_facecolor('#0d0d1a')
            ax[i].tick_params(colors=LC)

    # Channel config info
    ax[4].text(0.5, 0.5,
        f"Channel Config\n\nWBC → {results.get('best_wbc_name','?')}\nRBC → {results.get('best_rbc_name','?')}\n\n(Global learning)",
        ha='center', va='center', color=TC, fontsize=11,
        transform=ax[4].transAxes,
        bbox=dict(boxstyle='round', facecolor='#2a2a3e', alpha=0.8))
    ax[4].set_facecolor('#0d0d1a')
    ax[4].axis('off')
    row_label(0.150, "ROW 6\nSTATS", '#ffaa6b', '#3e2a1a')

    fig.suptitle("🔬 BLOOD CELL SEGMENTATION — PREPROCESSING PIPELINE",
                 color='#ffffff', fontsize=16, fontweight='bold', y=0.99,
                 bbox=dict(boxstyle='round', facecolor='#16213e', alpha=0.9))

    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  [Visualization] Saved: {save_path}")


## Section B — Global Channel Learning
Chạy **1 lần** trên toàn bộ train+val để tìm channel tốt nhất cho WBC/RBC.
Kết quả lưu vào `channel_config.json` và tái sử dụng mãi mà không cần annotation lúc inference.

In [46]:
# ============================================================
# SECTION B1: ANNOTATION PARSER
# ============================================================

def parse_ann(ann_path: str) -> list:
    """
    Đọc file annotation JSON của BCCD và trả về list bbox.
    Format BCCD: objects[].points.exterior = [[x1,y1],[x2,y2]]
    """
    if not ann_path or not os.path.exists(ann_path):
        print(f"  [WARN] Không tìm thấy: {ann_path}")
        return []
    with open(ann_path, "r", encoding="utf-8") as f:
        label_json = json.load(f)
    annotations = []
    for obj in label_json.get("objects", []):
        title  = obj.get("classTitle", "")
        points = obj.get("points", {}).get("exterior", [])
        if len(points) == 2:
            x1, y1 = points[0]
            x2, y2 = points[1]
            annotations.append({
                "title": title,
                "bbox": [min(x1,x2), min(y1,y2), abs(x2-x1), abs(y2-y1)]
            })
    return annotations


# ============================================================
# SECTION B2: COLOR SEPARATION ENGINE
# ============================================================

# Mapping: channel name → (mean_key trong color profile, index)
CHANNEL_DEFINITIONS = {
    "R":     ("mean_bgr",   2),  "G":     ("mean_bgr",   1),  "B":     ("mean_bgr",   0),
    "H":     ("mean_hsv",   0),  "S":     ("mean_hsv",   1),  "V":     ("mean_hsv",   2),
    "L":     ("mean_lab",   0),  "Lab_A": ("mean_lab",   1),  "Lab_B": ("mean_lab",   2),
    "Y":     ("mean_ycrcb", 0),  "Cr":    ("mean_ycrcb", 1),  "Cb":    ("mean_ycrcb", 2),
}


def _compute_color_profiles(all_class_pixels: dict) -> dict:
    """Tính mean BGR/LAB/HSV/YCrCb từ toàn bộ pixel đã gom của từng class."""
    profiles = {}
    for cls, px_list in all_class_pixels.items():
        all_px = np.vstack(px_list).astype(np.uint8)
        px_img = all_px.reshape(1, -1, 3)
        profiles[cls] = {
            "mean_bgr":   all_px.mean(axis=0),
            "mean_lab":   cv2.cvtColor(px_img, cv2.COLOR_BGR2LAB).reshape(-1,3).mean(axis=0),
            "mean_hsv":   cv2.cvtColor(px_img, cv2.COLOR_BGR2HSV).reshape(-1,3).mean(axis=0),
            "mean_ycrcb": cv2.cvtColor(px_img, cv2.COLOR_BGR2YCrCb).reshape(-1,3).mean(axis=0),
        }
    return profiles


def _best_channel_by_separation(color_profiles: dict, target_cls: str,
                                 other_classes: list, exclude: list = []) -> str:
    """
    Tìm channel có |mean_target - mean_others| lớn nhất.
    Channel này phân biệt target với các class còn lại tốt nhất → Otsu sẽ cắt chính xác hơn.
    """
    best_ch, best_sep = None, -np.inf
    target = color_profiles[target_cls]

    for ch_name, (mean_key, idx) in CHANNEL_DEFINITIONS.items():
        if ch_name in exclude:
            continue
        t_val = float(target[mean_key][idx])
        others = [float(color_profiles[c][mean_key][idx])
                  for c in other_classes if c in color_profiles]
        sep = abs(t_val - np.mean(others)) if others else abs(t_val - 128)
        if sep > best_sep:
            best_sep, best_ch = sep, ch_name

    return best_ch


# ============================================================
# SECTION B3: GLOBAL LEARNING — chạy 1 lần
# ============================================================

def learn_best_channels_from_dataset(
    dataset_dir: str,
    splits: list = ["train", "val"],
    save_path: str = CHANNEL_CONFIG_PATH
) -> dict:
    """
    Duyệt toàn bộ train+val, gom pixel từng class theo bbox annotation,
    tính màu trung bình toàn dataset, tìm channel có separation lớn nhất.

    Args:
        dataset_dir: Thư mục gốc BCCD dataset
        splits     : Các split dùng để học (không dùng test)
        save_path  : Nơi lưu channel_config.json

    Returns:
        dict vd: {"WBC": "B", "RBC": "Cr", "Platelets": "G"}
    """
    all_class_pixels = {}
    total_images = 0

    for split in splits:
        img_dir   = os.path.join(dataset_dir, split, "img")
        ann_dir   = os.path.join(dataset_dir, split, "ann")
        img_paths = sorted(glob(os.path.join(img_dir, "*.jpeg")) +
                           glob(os.path.join(img_dir, "*.jpg"))  +
                           glob(os.path.join(img_dir, "*.png")))
        print(f"  [{split}] {len(img_paths)} ảnh...")

        for img_path in img_paths:
            image    = cv2.imread(img_path)
            ann_path = os.path.join(ann_dir, os.path.basename(img_path) + ".json")
            if image is None or not os.path.exists(ann_path):
                continue

            annotations = parse_ann(ann_path)
            if not annotations:
                continue

            h_img, w_img = image.shape[:2]
            for ann in annotations:
                title        = ann["title"]
                x, y, bw, bh = [int(v) for v in ann["bbox"]]
                x1, y1 = max(0, x+2),      max(0, y+2)
                x2, y2 = min(w_img, x+bw-2), min(h_img, y+bh-2)
                crop = image[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                all_class_pixels.setdefault(title, []).append(crop.reshape(-1, 3))

            total_images += 1

    print(f"\n  Tổng: {total_images} ảnh đã xử lý")
    for cls, px_list in all_class_pixels.items():
        print(f"  {cls}: {sum(len(p) for p in px_list):,} pixels từ {len(px_list)} crops")

    print("\n  Tính color profiles toàn dataset...")
    color_profiles = _compute_color_profiles(all_class_pixels)

    print("  Tìm channel tốt nhất theo color separation...")
    all_classes = list(color_profiles.keys())
    result = {}
    for target_cls in all_classes:
        others = [c for c in all_classes if c != target_cls]
        best   = _best_channel_by_separation(color_profiles, target_cls, others)
        result[target_cls] = best
        print(f"  {target_cls:12s} → {best}")

    # Đảm bảo WBC ≠ RBC
    if result.get("WBC") == result.get("RBC"):
        wbc_ch = result["WBC"]
        print(f"  [WARN] WBC = RBC = '{wbc_ch}' → tìm channel thứ 2 cho RBC")
        result["RBC"] = _best_channel_by_separation(
            color_profiles, "RBC",
            [c for c in all_classes if c != "RBC"],
            exclude=[wbc_ch]
        )
        print(f"  RBC (adjusted) → {result['RBC']}")

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"\n  ✓ Config: {result}")
    print(f"  ✓ Lưu tại: {save_path}")
    return result


In [47]:
# ── Chạy 1 lần để tạo channel_config.json ─────────────────────────────────────
# Sau khi có file rồi, comment block này và chỉ chạy block load bên dưới.

print("=" * 60)
print("  GLOBAL CHANNEL LEARNING")
print("=" * 60)

CHANNEL_CONFIG = learn_best_channels_from_dataset(
    dataset_dir = DATASET_PATH,
    splits      = ["train", "val"],
    save_path   = CHANNEL_CONFIG_PATH,
)

# ── Hoặc load lại nếu đã có file ───────────────────────────────────────────────
# with open(CHANNEL_CONFIG_PATH) as f:
#     CHANNEL_CONFIG = json.load(f)

print(f"\nChannel config: {CHANNEL_CONFIG}")


  GLOBAL CHANNEL LEARNING
  [train] 205 ảnh...
  [val] 87 ảnh...

  Tổng: 292 ảnh đã xử lý
  WBC: 10,301,956 pixels từ 301 crops
  RBC: 32,501,399 pixels từ 3348 crops
  Platelets: 412,950 pixels từ 292 crops

  Tính color profiles toàn dataset...
  Tìm channel tốt nhất theo color separation...
  WBC          → S
  RBC          → B
  Platelets    → G

  ✓ Config: {'WBC': 'S', 'RBC': 'B', 'Platelets': 'G'}
  ✓ Lưu tại: /kaggle/working/channel_config.json

Channel config: {'WBC': 'S', 'RBC': 'B', 'Platelets': 'G'}


## Section C — Main Pipeline
`preprocess_image()` dùng `CHANNEL_CONFIG` đã học — không cần annotation lúc inference.

In [48]:
# ============================================================
# SECTION C: MAIN PIPELINE
# ============================================================

def preprocess_image(image_path: str = None) -> dict:
    """
    Pipeline tiền xử lý ảnh máu hoàn chỉnh (WBC + RBC segmentation).

    Args:
        image_path: Đường dẫn ảnh (JPG/PNG/BMP).
                    Truyền None để dùng ảnh demo tổng hợp.

    Returns:
        dict chứa: original | wbc_mask | rbc_mask | overlay | processed
                   + wbc_count | rbc_count | best_wbc_channel | best_rbc_channel
                   + các kết quả trung gian để debug/research
    """
    print("=" * 70)
    print("  BLOOD CELL SEGMENTATION PIPELINE")
    print("=" * 70)

    # ── STEP 1: Đọc ảnh ────────────────────────────────────────────────────
    print("\n[STEP 1] Đọc ảnh...")
    if image_path is None or not os.path.exists(str(image_path)):
        print("  → Không tìm thấy ảnh. Dùng ảnh demo tổng hợp...")
        image = create_synthetic_blood_image(512)
    else:
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Không thể đọc ảnh: {image_path}")
        print(f"  → OK: {image.shape} | {image_path}")

    original    = image.copy()
    h, w        = image.shape[:2]
    kernel_size = adaptive_kernel_size(image)
    print(f"  → {w}x{h} | kernel_size={kernel_size}")

    # ── STEP 2: Khử nhiễu ──────────────────────────────────────────────────
    print("\n[STEP 2] Khử nhiễu (4 phương pháp)...")
    denoise_results = denoise_image(image, kernel_size)
    denoised        = denoise_results["best"]
    denoise_method  = denoise_results["best_method_name"]

    # ── STEP 3: Color space analysis ───────────────────────────────────────
    print("\n[STEP 3] Phân tích color spaces...")
    cs_data  = analyze_color_spaces(denoised)
    channels = cs_data["channels"]

    # ── STEP 4: Load channel config (đã học từ train+val) ─────────────────
    print("\n[STEP 4] Load channel config từ global learning...")
    best_wbc_name    = CHANNEL_CONFIG.get("WBC", "B")    # fallback an toàn
    best_rbc_name    = CHANNEL_CONFIG.get("RBC", "Cr")
    best_wbc_channel = channels[best_wbc_name]
    best_rbc_channel = channels[best_rbc_name]
    channel_analysis = {
        "best_wbc_channel": best_wbc_name,
        "best_rbc_channel": best_rbc_name,
        "wbc_scores": {}, "rbc_scores": {},
        "wbc_ranking": [], "rbc_ranking": [],
    }
    print(f"  WBC → {best_wbc_name} | RBC → {best_rbc_name}")

    # ── STEP 5: Tăng tương phản ────────────────────────────────────────────
    print("\n[STEP 5] Tăng tương phản (CLAHE + Sharpening)...")
    contrast_wbc = enhance_contrast(best_wbc_channel)
    contrast_rbc = enhance_contrast(best_rbc_channel)
    clahe_wbc    = contrast_wbc["clahe_sharp"]
    clahe_rbc    = contrast_rbc["clahe_sharp"]

    # ── STEP 6: Segment WBC ────────────────────────────────────────────────
    print("\n[STEP 6] Phân đoạn bạch cầu (WBC)...")
    wbc_seg  = segment_wbc(clahe_wbc, kernel_size)
    wbc_mask = wbc_seg["final_mask"]

    # ── STEP 7: Loại WBC khỏi ảnh ─────────────────────────────────────────
    print("\n[STEP 7] Loại WBC khỏi ảnh (inpainting)...")
    overlay_data = create_wbc_overlay(denoised, wbc_mask, dilation_px=kernel_size // 2)
    image_no_wbc = overlay_data["image_no_wbc"]

    # ── STEP 8: Segment RBC — thử nhiều channel, chọn kết quả tốt nhất ───
    print("\n[STEP 8] Phân đoạn hồng cầu (RBC) — multi-channel selection...")
    cs_no_wbc = analyze_color_spaces(image_no_wbc)["channels"]

    # Danh sách channel ưu tiên thử cho RBC (theo thứ tự: config → Lab_A → R)
    # best_rbc_name từ global learning, thêm Lab_A và R để so sánh
    rbc_candidates = [best_rbc_name] + [
        ch for ch in ["Lab_A", "R", "Cr"] if ch != best_rbc_name
    ]

    best_rbc_seg, best_rbc_mask = None, None
    best_rbc_score, best_rbc_ch = -1.0, best_rbc_name

    for ch_name in rbc_candidates:
        if ch_name not in cs_no_wbc:
            continue
        ch_data  = cs_no_wbc[ch_name].astype(np.uint8)
        clahe_ch = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8,8)).apply(ch_data)
        seg      = segment_rbc(image_no_wbc, clahe_ch, kernel_size)
        score    = _score_rbc_mask(seg["final_mask"])
        print(f"    [{ch_name:6s}] count={score['count']:3d} | "
              f"circ={score['circularity']:.3f} | "
              f"unif={score['uniformity']:.3f} | "
              f"total={score['total']:.3f}"
              + (" ← best" if score["total"] > best_rbc_score else ""))
        if score["total"] > best_rbc_score:
            best_rbc_score = score["total"]
            best_rbc_seg   = seg
            best_rbc_mask  = seg["final_mask"]
            best_rbc_ch    = ch_name

    rbc_seg          = best_rbc_seg
    rbc_mask         = best_rbc_mask
    best_rbc_name    = best_rbc_ch   # cập nhật để vis hiển thị đúng
    best_rbc_channel = cs_no_wbc[best_rbc_name]
    print(f"  → RBC channel được chọn: {best_rbc_name} (score={best_rbc_score:.3f})")

    # ── STEP 9: Final overlay ──────────────────────────────────────────────
    print("\n[STEP 9] Tạo final overlay...")
    final_overlay = create_final_overlay(original, wbc_mask, rbc_mask)

    # ── STEP 10: Visualization ─────────────────────────────────────────────
    print("\n[STEP 10] Tạo pipeline visualization...")
    visualize_pipeline({
        "original":         original,
        "denoised":         denoised,
        "denoise_method":   denoise_method,
        "clahe_wbc":        clahe_wbc,
        "best_wbc_ch_img":  best_wbc_channel,
        "best_rbc_ch_img":  best_rbc_channel,
        "best_wbc_name":    best_wbc_name,
        "best_rbc_name":    best_rbc_name,
        "channels":         channels,
        "wbc_seg":          wbc_seg,
        "rbc_seg":          rbc_seg,
        "overlay_data":     overlay_data,
        "wbc_mask":         wbc_mask,
        "rbc_mask":         rbc_mask,
        "overlay":          final_overlay,
        "channel_analysis": channel_analysis,
    }, save_path=PIPELINE_FIG_PATH)

    # ── Tổng kết ───────────────────────────────────────────────────────────
    wbc_count = cv2.connectedComponentsWithStats(wbc_mask, connectivity=8)[0] - 1
    rbc_count = cv2.connectedComponentsWithStats(rbc_mask, connectivity=8)[0] - 1

    print("\n" + "=" * 70)
    print("  PIPELINE HOÀN TẤT!")
    print("=" * 70)
    print(f"  WBC channel  : {best_wbc_name}")
    print(f"  RBC channel  : {best_rbc_name}")
    print(f"  WBC phát hiện: {wbc_count}")
    print(f"  RBC phát hiện: {rbc_count}")
    print(f"  Denoise      : {denoise_method}")
    print(f"  Figure       : {PIPELINE_FIG_PATH}")
    print("=" * 70)

    return {
        # Kết quả chính
        "original":  original,
        "wbc_mask":  wbc_mask,
        "rbc_mask":  rbc_mask,
        "overlay":   final_overlay,
        "processed": image_no_wbc,
        # Thống kê
        "wbc_count":         wbc_count,
        "rbc_count":         rbc_count,
        "best_wbc_channel":  best_wbc_name,
        "best_rbc_channel":  best_rbc_name,
        "denoise_method":    denoise_method,
        # Trung gian (debug/research)
        "denoised":          denoised,
        "channels":          channels,
        "channel_analysis":  channel_analysis,
        "wbc_seg_steps":     wbc_seg,
        "rbc_seg_steps":     rbc_seg,
        "overlay_data":      overlay_data,
        "denoise_results":   denoise_results,
        "contrast_wbc":      contrast_wbc,
        "contrast_rbc":      contrast_rbc,
        "pipeline_fig":      PIPELINE_FIG_PATH,
    }


In [49]:
# ── Chạy pipeline trên ảnh mẫu ────────────────────────────────────────────────
result = preprocess_image(SAMPLE_IMG)

print(f"\n✓ WBC mask : {result['wbc_mask'].shape}")
print(f"✓ RBC mask : {result['rbc_mask'].shape}")
print(f"✓ WBC count: {result['wbc_count']}")
print(f"✓ RBC count: {result['rbc_count']}")
print(f"✓ Figure   : {result['pipeline_fig']}")


  BLOOD CELL SEGMENTATION PIPELINE

[STEP 1] Đọc ảnh...
  → OK: (480, 640, 3) | /kaggle/input/datasets/orvile/bccd-blood-cell-count-and-detection-dataset/train/img/BloodImage_00001.jpeg
  → 640x480 | kernel_size=5

[STEP 2] Khử nhiễu (4 phương pháp)...
  [Denoise] Noise level: 9.1 | Best: bilateral (score=0.279)

[STEP 3] Phân tích color spaces...

[STEP 4] Load channel config từ global learning...
  WBC → S | RBC → B

[STEP 5] Tăng tương phản (CLAHE + Sharpening)...
  [Contrast] Gamma tự động: 0.60 | CLAHE clipLimit=2.0 tileGrid=(8,8)
  [Contrast] Gamma tự động: 1.50 | CLAHE clipLimit=2.0 tileGrid=(8,8)

[STEP 6] Phân đoạn bạch cầu (WBC)...
  [WBC Seg] Otsu threshold: 114.0
  [WBC Seg] Phát hiện: 8 WBC

[STEP 7] Loại WBC khỏi ảnh (inpainting)...

[STEP 8] Phân đoạn hồng cầu (RBC) — multi-channel selection...
  [RBC Seg] Otsu threshold: 115.0
  [RBC Seg] Sau size filter: 41 RBC
    [B     ] count= 16 | circ=0.531 | unif=0.300 | total=0.304 ← best
  [RBC Seg] Otsu threshold: 121.0
  [RB